<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
print("DuckDB ready with HF secret. Developing against month =", MONTH)

DuckDB ready with HF secret. Developing against month = 2026-03


In [2]:
cols_fact = con.sql(f"""
    SELECT * FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    LIMIT 5
""").df()
print("fact_content_daily_performance columns:", cols_fact.columns.tolist())
cols_fact.head()

fact_content_daily_performance columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content page (`content_hash_id`) on one day
(`report_date`) — one row in `fact_content_daily_performance` is a single
page-day observation.

**Time window:** developing against `month=2026-03`, a mid-panel month. Never
using the `_sample` table for label logic — it holds the sealed final month
(June 2026), the natural outcome window for any past→future label.

**Table used:** `fact_content_daily_performance` only. `dim_content` isn't
needed for this contract — every feature and the label proxy can be built
directly from the daily fact table's own columns.

**What I'd predict/rank:** a refresh-priority score per page, using GSC and GA4
signals observed up to the decision point.

**One thing I deliberately exclude:** any rebuilt version of FlyRank's own
`health_score`, `priority_score`, or `action_type` — these are product
decisions, not observable signals, and using them as features would just teach
a model to copy the existing rule instead of finding real signal.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Unit of analysis: one row = one (content_hash_id, report_date) pair")
print(f"Developing against: month={MONTH}")

Unit of analysis: one row = one (content_hash_id, report_date) pair
Developing against: month=2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (knowable before the decision point):** gsc_impressions, gsc_clicks,
gsc_avg_position, ga4_sessions, ga4_engaged_sessions.

**Label/proxy:** same-window decline bucket for now (starter-style proxy); a
stronger future version — decline over a following window — needs multi-day
ordering from this same table.

**Context (not features, not labels):** content_hash_id, client_hash_id,
report_date, month — join keys and identifiers, not model inputs on their own.

**Excluded, with why:** health_score / priority_score / action_type — product
decisions, not observable signals (would create a circular result if used as
features). Also excluding the availability flags (`client_has_gsc`,
`client_has_ga4`, `gsc_data_available`, `ga4_data_available`) as *features* —
these are filters for whether a row is usable, not performance signal about the
page itself.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

buckets = {
    "features": ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions", "ga4_engaged_sessions"],
    "label_or_proxy": ["same-window decline bucket (proxy); future-window decline (target)"],
    "context": ["content_hash_id", "client_hash_id", "report_date", "month"],
    "excluded": ["health_score", "priority_score", "action_type",
                 "client_has_gsc", "client_has_ga4", "gsc_data_available", "ga4_data_available (as features)"],
}
for k, v in buckets.items():
    print(f"{k}: {v}")

features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']
label_or_proxy: ['same-window decline bucket (proxy); future-window decline (target)']
context: ['content_hash_id', 'client_hash_id', 'report_date', 'month']
excluded: ['health_score', 'priority_score', 'action_type', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available (as features)']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1 : grain

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicate (content_hash_id, report_date) pairs: {len(grain_check)}")
print("Zero confirms the grain: one row really is one page-day.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (content_hash_id, report_date) pairs: 0
Zero confirms the grain: one row really is one page-day.


Query 2 : row count + date span

In [8]:
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS n_pages
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
span

,n_rows,min_date,max_date,n_pages
0,9841378,2026-03-01,2026-03-31,331437


Query 3 : availability with IS TRUE

In [9]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
print(availability)
print(f"{availability['ga4_available_rows'][0]} of {availability['total_rows'][0]} rows have GA4 data — "
      f"the rest are search-only rows from clients whose GA4 tracking hadn't started.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378              413966
413966 of 9841378 rows have GA4 data — the rest are search-only rows from clients whose GA4 tracking hadn't started.


Five features, each with when it's knowable:

1. **gsc_impressions** — observed Search Console data up to that report_date,
   no future data involved.
2. **gsc_clicks** — same: observed same-day search-console data.
3. **gsc_avg_position** — observed ranking position for that day, known as it happens.
4. **ga4_sessions** — observed GA4 sessions for that day (where `ga4_data_available IS TRUE`).
5. **ga4_engaged_sessions** — observed engagement count for that day, same-day, no lookahead.

## build the feature frame:

In [10]:
features_df = con.sql(f"""
    SELECT content_hash_id, report_date, gsc_impressions, gsc_clicks,
           gsc_avg_position, ga4_sessions, ga4_engaged_sessions
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE ga4_data_available IS TRUE
    LIMIT 2000
""").df()
features_df.head()

,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,content_09be8cc7fcb222af,2026-03-01,0,0,NaN,1,0
1,content_851afac9fe13612e,2026-03-01,0,0,NaN,1,0
2,content_cee6c6fc8c51af14,2026-03-01,0,0,NaN,1,0
3,content_5e120e972f11f833,2026-03-01,0,0,NaN,1,0
4,content_16a7291bb6ecaebe,2026-03-01,0,0,NaN,1,0


Deliberately injecting a label-derived column to watch the score jump toward
perfect, then removing it — the Notebook 2 leakage lesson, reproduced here on
real warehouse data.

##  spring the trap, then remove it:

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

X = features_df[["gsc_impressions", "gsc_clicks", "gsc_avg_position",
                   "ga4_sessions", "ga4_engaged_sessions"]].fillna(0)
y = (features_df["gsc_clicks"] < features_df["gsc_clicks"].median()).astype(int)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
honest_score = precision_score(y, honest_model.predict(X))
print(f"Honest Precision: {honest_score:.3f}")

X_leaky = X.copy()
X_leaky["leaky_col"] = y

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_score = precision_score(y, leaky_model.predict(X_leaky))
print(f"Leaky Precision (jumps toward perfect): {leaky_score:.3f}")

del X_leaky
print(f"\nLeaky column removed. Keeping the honest number: {honest_score:.3f}")

Honest Precision: 1.000
Leaky Precision (jumps toward perfect): 1.000

Leaky column removed. Keeping the honest number: 1.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data cannot tell me:
- **Causation** — whether refreshing a page causes recovery; that needs an
  actual experiment, not observational daily facts.
- Anything about clients whose GA4 tracking hadn't started — those rows are
  search-only (`ga4_data_available = FALSE`), so early history is incomplete by
  design, not by data quality issues.
- History depth is **unbalanced across clients** — different clients have
  different amounts of history, so cross-client comparisons need to account for
  this rather than assume equal history everywhere.
- Anything about real client identities, domains, or queries — all identifiers
  are pseudonymized hashes.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

ga4_missing_pct = (1 - availability['ga4_available_rows'][0] / availability['total_rows'][0]) * 100
print(f"{ga4_missing_pct:.1f}% of rows in this month lack GA4 data — a real limit on what this slice can show.")

95.8% of rows in this month lack GA4 data — a real limit on what this slice can show.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.